# Age and Gender Distortion

In a [recent paper in Nature](https://www.nature.com/articles/s41586-025-09581-z), Douglas Guilbeault, Solène Delecourt & Bhargav Srinivasa Desikan investigated the effects of age distortion on genders.

In this assignent, you will work through some of their results yoursefl. You will use their data, available at <https://github.com/drguilbe/distortion_age_gender_online/>.

## Setup

**Python.** Use Python 3.10+. Create a virtual environment and install dependencies so anyone can rerun the notebook:

```bash
python3 -m venv .venv
source .venv/bin/activate 
pip install -r requirements.txt
```

**Packages** (see `requirements.txt`): `pandas`, `numpy`, `matplotlib`, `seaborn`, `statsmodels`, `scipy`, `pingouin`, `scipy.stats`, `plotly`, `jupyter`, `ipykernel`.


## Part 1 - Correlation between age and gender

We load `GPT2-large-dimensions.csv`. The file includes **three extraction methods** for each construct, following the paper’s robustness checks:

| Construct | Columns (raw scores, not min–max normalized) |
|-----------|-----------------------------------------------|
| Age | `age.main`, `age.ext`, `age.red` |
| Gender | `gender.main`, `gender.ext`, `gender.red` |

The **primary** Pearson correlation matches the replication R script (`fig2_GPT2-large_analyses.R`): **`age.main`** vs **`gender.main`**. Companion columns `*_norm.*` are min–max normalized versions used elsewhere, we use the raw `age.*` / `gender.*` triples for the main correlation and for the robustness heatmaps.

On the **heatmaps**, the main method is labeled **`age_score`** / **`gender_score`** (same values as `age.main` / `gender.main`). Axes follow **`red`, `ext`, `score`** order to match the assignment figures. The **color bars differ**: the age heatmap uses **0.60–1.00**; the gender heatmap uses **0.75–1.00** (both with 0.05 tick spacing within that range). A **combined 6×6** heatmap (`correlation_heatmap.svg`) correlates all six measures (`age_red` … `gender_score`) with a **0.20–1.00** scale (ticks every 0.1).

The summary table uses **Pingouin**’s `corr` (Pearson *r*, 95% CI, *p*, Bayes factor BF10, post-hoc power) so the printed output aligns with the assignment example.

In [50]:
import numpy as np
from pathlib import Path

from matplotlib.cm import ScalarMappable
from matplotlib.colors import LinearSegmentedColormap, Normalize
from mpl_toolkits.axes_grid1 import make_axes_locatable


import matplotlib.pyplot as plt
import pandas as pd
import pingouin as pg
import re
import seaborn as sns


BASE = Path.cwd()
DATA_PATH = BASE / "GPT2-large-dimensions.csv"

df = pd.read_csv(DATA_PATH)

# Theme (heatmap figures)
_BG = "#EDEDED"
_LABEL = "#000000"
_GRID = "#FFFFFF"
_CELL = "#FFFFFF"

# Colormap
_CORR_CMAP_COLORS = [
    "#F2B49A",
    "#E9A07E",
    "#E88A6A",
    "#E46F5A",
    "#DD4B39",
    "#D90A3A",
    "#C8002E",
]


def correlation_colormap() -> LinearSegmentedColormap:
    return LinearSegmentedColormap.from_list("corr_red", _CORR_CMAP_COLORS, N=256)


# Color scale and legend ticks
_COLORBAR_AGE_VMIN, _COLORBAR_AGE_VMAX = 0.6, 1.0
_COLORBAR_AGE_TICKS = np.linspace(_COLORBAR_AGE_VMIN, _COLORBAR_AGE_VMAX, 9)

_COLORBAR_GENDER_VMIN, _COLORBAR_GENDER_VMAX = 0.75, 1.0
_COLORBAR_GENDER_TICKS = np.linspace(_COLORBAR_GENDER_VMIN, _COLORBAR_GENDER_VMAX, 6)

# Combined age + gender 
_CROSS_COLS = [
    "age.red",
    "age.ext",
    "age.main",
    "gender.red",
    "gender.ext",
    "gender.main",
]
_CROSS_LABELS = [
    "age_red",
    "age_ext",
    "age_score",
    "gender_red",
    "gender_ext",
    "gender_score",
]
_COLORBAR_FULL_VMIN, _COLORBAR_FULL_VMAX = 0.2, 1.0
_COLORBAR_FULL_TICKS = np.linspace(_COLORBAR_FULL_VMIN, _COLORBAR_FULL_VMAX, 9)


def pairwise_corr_matrix(
    frame: pd.DataFrame,
    *,
    cols: list[str],
    labels: list[str],
) -> pd.DataFrame:
    sub = frame[cols].dropna()
    r = sub.corr(method="pearson").reindex(index=cols, columns=cols)
    r.index = labels
    r.columns = labels
    return r


def plot_correlation_heatmap(
    corr: pd.DataFrame,
    *,
    out_path: Path,
    vmin: float,
    vmax: float,
    cbar_ticks: np.ndarray,
    figsize: tuple[float, float] = (4.5, 3.8),
    xtick_rotation: float = 0.0,
    annot_fontsize: float = 10.0,
    annot_fmt: str = ".2f",
    cbar_fmt: str = "%.2f",
) -> None:
    vmin_val, vmax_val = vmin, vmax
    cmap = correlation_colormap()
    norm = Normalize(vmin=vmin_val, vmax=vmax_val)

    fig, ax = plt.subplots(figsize=figsize)
    fig.patch.set_facecolor(_BG)
    ax.set_facecolor(_BG)

    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size="4.5%", pad=0.12)
    cax.set_facecolor(_BG)

    sns.heatmap(
        corr,
        annot=False,
        vmin=vmin_val,
        vmax=vmax_val,
        cmap=cmap,
        square=True,
        linewidths=0.5,
        linecolor=_GRID,
        ax=ax,
        cbar=False,
    )
    sm = ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])
    cb = fig.colorbar(sm, cax=cax, ticks=cbar_ticks, format=cbar_fmt)
    cb.ax.tick_params(colors=_LABEL)
    cb.outline.set_visible(False)

    ax.set_title("Correlation Heatmap", fontsize=12, color=_LABEL)
    ax.tick_params(axis="both", colors=_LABEL)
    if xtick_rotation:
        plt.setp(ax.get_xticklabels(), rotation=xtick_rotation, ha="right")
    else:
        plt.setp(ax.get_xticklabels(), rotation=0, ha="center")

    # Cell values: white on colored tiles
    for i in range(corr.shape[0]):
        for j in range(corr.shape[1]):
            val = float(corr.iloc[i, j])
            ax.text(
                j + 0.5,
                i + 0.5,
                f"{val:{annot_fmt}}",
                ha="center",
                va="center",
                color=_CELL,
                fontsize=annot_fontsize,
            )

    plt.tight_layout()
    fig.savefig(out_path, format="svg", bbox_inches="tight", facecolor=_BG)
    plt.close(fig)


# Axis order and *_score labels match the assignment figures
age_r = pairwise_corr_matrix(
    df,
    cols=["age.red", "age.ext", "age.main"],
    labels=["age_red", "age_ext", "age_score"],
)
gender_r = pairwise_corr_matrix(
    df,
    cols=["gender.red", "gender.ext", "gender.main"],
    labels=["gender_red", "gender_ext", "gender_score"],
)

plot_correlation_heatmap(
    age_r,
    out_path=BASE / "correlation_heatmap_age.svg",
    vmin=_COLORBAR_AGE_VMIN,
    vmax=_COLORBAR_AGE_VMAX,
    cbar_ticks=_COLORBAR_AGE_TICKS,
)
plot_correlation_heatmap(
    gender_r,
    out_path=BASE / "correlation_heatmap_gender.svg",
    vmin=_COLORBAR_GENDER_VMIN,
    vmax=_COLORBAR_GENDER_VMAX,
    cbar_ticks=_COLORBAR_GENDER_TICKS,
)

# Full cross-correlation: all six age + gender measures
cross_r = pairwise_corr_matrix(df, cols=_CROSS_COLS, labels=_CROSS_LABELS)
plot_correlation_heatmap(
    cross_r,
    out_path=BASE / "correlation_heatmap.svg",
    vmin=_COLORBAR_FULL_VMIN,
    vmax=_COLORBAR_FULL_VMAX,
    cbar_ticks=_COLORBAR_FULL_TICKS,
    figsize=(8.8, 7.2),
    annot_fontsize=9.0,
    cbar_fmt="%.1f",
)


In [51]:
from pathlib import Path

from IPython.display import HTML, display

BASE = Path.cwd()
_row = (
    '<div style="display:flex; gap:16px; align-items:flex-start; flex-wrap:wrap;">'
    f'<div style="flex:0 0 auto;">{(BASE / "correlation_heatmap_age.svg").read_text()}</div>'
    f'<div style="flex:0 0 auto;">{(BASE / "correlation_heatmap_gender.svg").read_text()}</div>'
    "</div>"
)
_full = (
    '<div style="margin-top:16px; max-width:100%;">'
    f'{(BASE / "correlation_heatmap.svg").read_text()}'
    "</div>"
)
display(HTML(_row + _full))

## Part 2 - Relationship between age and gender

We use the **min-max normalized** main measures in `GPT2-large-dimensions.csv`: `age_norm.main` and `gender_norm.main`. Run OLS, check **R^2 ≈ 0.761** and coefficients against the reference table below, draw a **static** scatter with regression line and the same **representative categories** as `age_gender_regression_plot.svg`, then an **interactive** Plotly figure saved as `age_gender_regression_interactive.html`.

Reference OLS summary (from the assignment):

| | |
|--|--|
| Dep. Variable | `age_norm_main` |
| R-squared | 0.761 |
| Intercept | −0.0211 |
| `gender_norm_main` | 0.7450 |
| *n* | 3495 |

In [52]:
import statsmodels.formula.api as smf
import html

from IPython.display import HTML, display

import plotly.graph_objects as go
from matplotlib.ticker import FixedLocator, FormatStrFormatter, NullLocator

#CSV column names vs paper
COL_AGE = "age_norm.main"
COL_GENDER = "gender_norm.main"
assert COL_AGE in df.columns and COL_GENDER in df.columns, (
    f"Expected {COL_AGE!r} and {COL_GENDER!r}; got {list(df.columns)}"
)

reg = df.rename(
    columns={"Social.Category": "category", COL_AGE: "age_norm_main", COL_GENDER: "gender_norm_main"}
).dropna(subset=["age_norm_main", "gender_norm_main"])

ols = smf.ols("age_norm_main ~ gender_norm_main", data=reg).fit()

_OLS_SUMMARY_CSS = """
<style>
.ols-dark-summary {
  --ols-bg: var(--vscode-editor-background, var(--jp-layout-color0, #1e1e1e));
  background: var(--ols-bg);
  color: #fff;
  padding: 12px 16px;
  display: inline-block;
  font-family: system-ui, -apple-system, Segoe UI, Helvetica, Arial, sans-serif;
  font-size: 13px;
  line-height: 1.35;
}
.ols-dark-summary table.simpletable {
  border-collapse: collapse;
  background: var(--ols-bg);
  color: #fff;
  margin: 0;
}
.ols-dark-summary table.simpletable caption {
  caption-side: top;
  text-align: center;
  color: #fff;
  font-weight: bold;
  padding: 8px 4px 12px;
  background: var(--ols-bg);
}
.ols-dark-summary table.simpletable th,
.ols-dark-summary table.simpletable td {
  border: 1px solid #fff;
  color: #fff;
  background: var(--ols-bg);
  padding: 4px 10px;
  text-align: left;
  vertical-align: middle;
}
.ols-dark-summary table.simpletable th {
  font-weight: bold;
}
</style>
"""
_ols_html = ols.summary().as_html()
if "<br/><br/>Notes:" in _ols_html:
    _ols_html = _ols_html.split("<br/><br/>Notes:", 1)[0]
display(
    HTML(
        _OLS_SUMMARY_CSS
        + '<div class="ols-dark-summary">'
        + _ols_html
        + "</div>"
    )
)

assert np.isclose(ols.rsquared, 0.761, atol=1e-3), ols.rsquared
assert np.isclose(ols.params["Intercept"], -0.0211, atol=5e-4)
assert np.isclose(ols.params["gender_norm_main"], 0.7450, atol=5e-4)

#Static plot: same categories as age_gender_regression_plot.svg
REPRESENTATIVE_GOLD = [
    "cook",
    "homoeopath",
    "intern",
    "novice",
    "secretary",
]
REPRESENTATIVE_BLUE = [
    "chairman of the board",
    "chief of staff",
    "director of research",
    "elected official",
    "military personnel",
]
REPRESENTATIVE = REPRESENTATIVE_GOLD + REPRESENTATIVE_BLUE
_COL_GOLD = "#DDB115"
_COL_BLUE = "#6EB3E0"
_LINE_RED = "#E02020"
# Plotly strips most HTML in hovers
def _plotly_hover_row(r):
    c = html.escape(str(r["category"]))
    return (
        f"<i>social_category</i> <b>{c}</b><br>"
        f"<i>gender_norm</i> <b>{r['gender_norm_main']:.2f}</b><br>"
        f"<i>age_norm</i> <b>{r['age_norm_main']:.2f}</b>"
    )


# Label anchors in data coordinates (match reference ggplot screenshot layout)
_XLIM = (-0.25, 1.4)
_YLIM = (-0.2, 1.2)
_LABEL_STYLE = {
    "chairman of the board": {"xytext": (0.8, 0.9), "ha": "left", "va": "bottom"},
    "elected official": {"xytext": (0.7, 0.82), "ha": "right", "va": "bottom"},
    "military personnel": {"xytext": (0.48, 0.68), "ha": "right", "va": "center"},
    "director of research": {"xytext": (1.0, 0.48), "ha": "left", "va": "top"},
    "chief of staff": {"xytext": (0.92, 0.38), "ha": "left", "va": "top"},
    "homoeopath": {"xytext": (0.0, 0.27), "ha": "right", "va": "bottom"},
    "intern": {"xytext": (0.0, 0.16), "ha": "right", "va": "center"},
    "cook": {"xytext": (0.08, -0.06), "ha": "right", "va": "top"},
    "secretary": {"xytext": (0.42, -0.06), "ha": "left", "va": "top"},
    "novice": {"xytext": (0.52, 0.15), "ha": "left", "va": "center"},
}

# Static figure style aligned with reference age_gender_regression_plot.svg (gray face, white grid)
_FACE = "#ebebeb"
_GRID = "#ffffff"
_TICKLABEL = "#4d4d4d"
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["Helvetica", "Arial", "DejaVu Sans", "sans-serif"]
fig, ax = plt.subplots(figsize=(6.4, 4.8))
ax.set_facecolor(_FACE)
fig.patch.set_facecolor("#ffffff")
ax.set_axisbelow(True)
ax.grid(True, which="major", color=_GRID, linewidth=1.0, linestyle="-")
ax.xaxis.set_major_locator(FixedLocator([0.0, 0.5, 1.0]))
ax.xaxis.set_minor_locator(NullLocator())
ax.yaxis.set_major_locator(FixedLocator([0.0, 0.4, 0.8]))
ax.yaxis.set_minor_locator(NullLocator())

ax.scatter(
    reg["gender_norm_main"],
    reg["age_norm_main"],
    s=12,
    facecolors="none",
    edgecolors="#000000",
    linewidths=0.89,
    zorder=2,
)

_intercept = float(ols.params.Intercept)
_slope = float(ols.params["gender_norm_main"])
# Plotly draws the full line segment
_x_lo = 0.0 if _intercept >= 0 else max(0.0, -_intercept / _slope)
_x_hi = 1.0
if _intercept + _slope * _x_hi > 1.0:
    _x_hi = float((1.0 - _intercept) / _slope)
# Plotly OLS: extend along y = a + b x into negative x/y (like mpl clip_on=False), then clip in [0,1]^2
_plotly_x_ext = -0.02
_n_ext = 40
gx_ext = np.linspace(_plotly_x_ext, _x_lo, _n_ext)
y_ext = _intercept + _slope * gx_ext
gx_in = np.linspace(_x_lo, _x_hi, 100)
y_in = np.clip(_intercept + _slope * gx_in, 0.0, 1.0)
gx_line_plotly = np.concatenate([gx_ext, gx_in[1:]])
yy_line_plotly = np.concatenate([y_ext, y_in[1:]])

_gx_mpl = np.linspace(_XLIM[0], _XLIM[1], 200)
_yy_mpl = _intercept + _slope * _gx_mpl
ax.plot(_gx_mpl, _yy_mpl, color=_LINE_RED, linewidth=2.0, zorder=3, clip_on=False)

for cats, col in ((REPRESENTATIVE_GOLD, _COL_GOLD), (REPRESENTATIVE_BLUE, _COL_BLUE)):
    sub = reg[reg["category"].isin(cats)]
    ax.scatter(
        sub["gender_norm_main"],
        sub["age_norm_main"],
        s=48,
        facecolors=col,
        zorder=5,
        edgecolors="#000000",
        linewidths=0.89,
    )

for cat in REPRESENTATIVE:
    row = reg[reg["category"] == cat]
    if row.empty:
        continue
    x = float(row["gender_norm_main"].iloc[0])
    y = float(row["age_norm_main"].iloc[0])
    st = _LABEL_STYLE[cat]
    ax.annotate(
        cat,
        xy=(x, y),
        xytext=st["xytext"],
        textcoords="data",
        fontsize=8,
        arrowprops=dict(arrowstyle="-", color="gray", lw=0.6, shrinkA=0, shrinkB=2),
        ha=st["ha"],
        va=st["va"],
        clip_on=False,
    )

ax.set_xlabel(
    "Gender Association\n(Female-Male Dimension)",
    fontweight="bold",
    fontsize=12,
    color=_TICKLABEL,
)
ax.set_ylabel(
    "Age Association\n(Young-Old Dimension)",
    fontweight="bold",
    fontsize=12,
    color=_TICKLABEL,
)
ax.xaxis.set_major_formatter(FormatStrFormatter("%.1f"))
ax.yaxis.set_major_formatter(FormatStrFormatter("%.1f"))
ax.tick_params(
    axis="both",
    which="major",
    colors=_TICKLABEL,
    labelsize=10,
    direction="out",
    length=5,
    width=1.5,
    bottom=True,
    top=False,
    left=True,
    right=False,
)
for _side in ("bottom", "left", "top", "right"):
    ax.spines[_side].set_visible(False)
ax.set_xlim(_XLIM)
ax.set_ylim(_YLIM)
ax.margins(0)
ax.set_autoscale_on(False)
ax.set_aspect("equal", adjustable="box")
fig.tight_layout()
fig.savefig(BASE / "age_gender_regression_plot.svg", format="svg", bbox_inches="tight")
plt.close(fig)

reg["_plotly_hover"] = reg.apply(_plotly_hover_row, axis=1)

#Interactive Plotly (
reg_gold = reg[reg["category"].isin(REPRESENTATIVE_GOLD)]
reg_blue = reg[reg["category"].isin(REPRESENTATIVE_BLUE)]
fig_p = go.Figure()
fig_p.add_trace(
    go.Scatter(
        x=reg["gender_norm_main"],
        y=reg["age_norm_main"],
        mode="markers",
        marker=dict(
            size=7,
            opacity=0.22,
            color="#A8A8A8",
            line=dict(width=0.9, color="#3A3A3A"),
        ),
        hovertext=reg["_plotly_hover"],
        hovertemplate="%{hovertext}<extra></extra>",
        showlegend=False,
        cliponaxis=False,
    )
)
fig_p.add_trace(
    go.Scatter(
        x=reg_gold["gender_norm_main"],
        y=reg_gold["age_norm_main"],
        mode="markers",
        marker=dict(size=13, color=_COL_GOLD, line=dict(width=1.2, color="#3A3A3A")),
        hovertext=reg_gold["_plotly_hover"],
        hovertemplate="%{hovertext}<extra></extra>",
        showlegend=False,
        cliponaxis=False,
    )
)
fig_p.add_trace(
    go.Scatter(
        x=reg_blue["gender_norm_main"],
        y=reg_blue["age_norm_main"],
        mode="markers",
        marker=dict(size=13, color=_COL_BLUE, line=dict(width=1.2, color="#3A3A3A")),
        hovertext=reg_blue["_plotly_hover"],
        hovertemplate="%{hovertext}<extra></extra>",
        showlegend=False,
        cliponaxis=False,
    )
)
fig_p.add_trace(
    go.Scatter(
        x=gx_line_plotly,
        y=yy_line_plotly,
        mode="lines",
        line=dict(color=_LINE_RED, width=2.5, simplify=False),
        cliponaxis=False,
        hoverinfo="skip",
        showlegend=False,
    )
)
fig_p.update_layout(
    width=750,
    height=650,
    margin=dict(l=60, r=40, t=40, b=60),
    xaxis_title="<b>Gender Association</b><br><b>(Female-Male Dimension)</b>",
    yaxis_title="<b>Age Association</b><br><b>(Young-Old Dimension)</b>",
    xaxis_title_font=dict(size=13, color="black", family="Arial Black, Helvetica Neue, Helvetica, sans-serif"),
    yaxis_title_font=dict(size=13, color="black", family="Arial Black, Helvetica Neue, Helvetica, sans-serif"),
    xaxis=dict(
        range=[0, 1],
        autorange=False,
        constrain="domain",
        tickmode="linear",
        tick0=0,
        dtick=0.1,
        tickformat=".1f",
        tickfont=dict(size=11, color="black", family="Helvetica Neue, Helvetica, Arial, sans-serif"),
        ticks="outside",
        ticklen=5,
        tickwidth=1,
        tickcolor="#000000",
        showgrid=True,
        gridcolor="#D8D8D8",
        gridwidth=1,
        zeroline=False,
        showline=True,
        linewidth=1.25,
        linecolor="#000000",
        mirror=False,
    ),
    yaxis=dict(
        range=[0, 1],
        autorange=False,
        constrain="domain",
        tickmode="linear",
        tick0=0,
        dtick=0.1,
        tickformat=".1f",
        tickfont=dict(size=11, color="black", family="Helvetica Neue, Helvetica, Arial, sans-serif"),
        ticks="outside",
        ticklen=5,
        tickwidth=1,
        tickcolor="#000000",
        showgrid=True,
        gridcolor="#D8D8D8",
        gridwidth=1,
        zeroline=False,
        showline=True,
        linewidth=1.25,
        linecolor="#000000",
        mirror=False,
    ),
    shapes=[
        dict(
            type="line",
            xref="x",
            yref="y",
            x0=0,
            x1=1,
            y0=1,
            y1=1,
            layer="below",
            line=dict(color="#D8D8D8", width=1),
        ),
        dict(
            type="line",
            xref="x",
            yref="y",
            x0=1,
            x1=1,
            y0=0,
            y1=1,
            layer="below",
            line=dict(color="#D8D8D8", width=1),
        ),
    ],
    showlegend=False,
    template="plotly_white",
    plot_bgcolor="white",
    hoverlabel=dict(
        bgcolor="white",
        bordercolor="#D8D8D8",
        font_size=12,
        font_family="system-ui, -apple-system, BlinkMacSystemFont, Segoe UI, sans-serif",
    ),
)
fig_p.write_html(BASE / "age_gender_regression_interactive.html", include_plotlyjs="cdn")

**Outliers and what the figure highlights**

The static plot spotlights a small set of **illustrative** occupations (e.g. “chairman of the board”, “intern”, “cook”) that sit at informative positions along the gender–age gradient or illustrate familiar social roles. Many **other** points lie far from the regression line—categories with strong residual age for their gender score, or extreme on one axis. Those are equally “outliers” in a statistical sense, but naming every one would clutter the figure and dilute the narrative; authors typically **curate** labels for clarity. The interactive plot matters because **hover** reveals those unlabeled extremes (and mid-list categories), so readers can judge how much of the cloud is driven by a few distant points versus a smooth association—highlighting choices shape which deviations readers notice.